In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

import os
from dotenv import load_dotenv
load_dotenv(dotenv_path=project_root / ".env")

from src.retrieval.create_vectorstore import load_vectorstore
from src.rag.rag_chain import build_rag_chain

c:\Healthcare_Prior_Authorization_AI_Assistant\h1venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
vectorstore = load_vectorstore(str(project_root / "data" / "processed" / "faiss_index"))
rag_chain = build_rag_chain(vectorstore)
print(f"Loaded {vectorstore.index.ntotal} vectors")

c:\Healthcare_Prior_Authorization_AI_Assistant\src\retrieval\create_vectorstore.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  return HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3295.48it/s]


Vectorstore loaded from: c:\Healthcare_Prior_Authorization_AI_Assistant\data\processed\faiss_index
Loaded 1351 vectors


In [3]:
test_set = [
    # Spine (0236) - 3 questions
    ("What are the medical necessity criteria for MRI of the spine?", "0236", [2]),
    ("When is dynamic-kinetic MRI considered experimental?", "0236", [3]),
    ("When is MRI not medically necessary for spine trauma?", "0236", [3, 7]),
    
    # Knee (0673) - 2 questions
    ("When is knee arthroscopy medically necessary?", "0673", [2, 3]),
    ("What are the criteria for meniscal repair?", "0673", [2, 3, 4]),
    
    # Cardiac MRI (0520) - 2 questions
    ("What are the indications for cardiac MRI?", "0520", [2, 3, 14]),
    ("When is cardiac MRI used for pericardial disease?", "0520", [2, 3]),
    
    # MRCP (0384) - 2 questions
    ("When is MRCP appropriate for PSC?", "0384", [14, 31]),
    ("What is the role of MRCP versus ERCP?", "0384", [14, 31]),
    
    # Extremities (0171) - 2 questions
    ("What are the criteria for MRI of extremities?", "0171", [2, 3]),
    ("When is MRI used for diabetic foot ulcer?", "0171", [3]),
    
    # Out-of-scope - 1 question (should decline)
    ("Does Medicare cover MRI of the spine?", "DECLINE", []),
]

print(f"Total test questions: {len(test_set)}")

Total test questions: 12


In [4]:
import time
import pandas as pd

DECLINE_PHRASE = "The provided policy documents do not contain this information."

results = []

for i, (question, expected_policy, expected_pages) in enumerate(test_set):
    print(f"[{i+1}/{len(test_set)}] {question[:60]}...")
    
    try:
        # 1. Retrieval evaluation
        retrieved_docs = vectorstore.similarity_search(question, k=5)
        retrieved_policies = [d.metadata['policy_number'] for d in retrieved_docs]
        retrieved_pages = [d.metadata['page'] for d in retrieved_docs]
        
        # 2. LLM answer
        answer = rag_chain.invoke(question)
        declined = DECLINE_PHRASE in answer
        
        # 3. Grade retrieval
        if expected_policy == "DECLINE":
            # Out-of-scope: we don't care about retrieval, just that LLM declined
            correct_policy_retrieved = None
            correct_page_retrieved = None
            expected_behavior = declined
        else:
            correct_policy_retrieved = expected_policy in retrieved_policies
            correct_page_retrieved = any(
                p in expected_pages 
                for p, pol in zip(retrieved_pages, retrieved_policies) 
                if pol == expected_policy
            )
            expected_behavior = not declined  # should have answered
        
        results.append({
            "question": question,
            "expected_policy": expected_policy,
            "expected_pages": expected_pages,
            "retrieved_policies": retrieved_policies,
            "retrieved_pages": retrieved_pages,
            "correct_policy_in_top5": correct_policy_retrieved,
            "correct_page_in_top5": correct_page_retrieved,
            "declined": declined,
            "expected_behavior_ok": expected_behavior,
            "answer": answer
        })
        
        # Rate-limit friendly pause
        time.sleep(2)
    
    except Exception as e:
        print(f"  ERROR: {e}")
        results.append({
            "question": question,
            "expected_policy": expected_policy,
            "expected_pages": expected_pages,
            "error": str(e)
        })
        time.sleep(5)

print("\nEvaluation complete!")
df = pd.DataFrame(results)

[1/12] What are the medical necessity criteria for MRI of the spine...


c:\Healthcare_Prior_Authorization_AI_Assistant\h1venv\lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[2/12] When is dynamic-kinetic MRI considered experimental?...


c:\Healthcare_Prior_Authorization_AI_Assistant\h1venv\lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[3/12] When is MRI not medically necessary for spine trauma?...


...

[11/12] When is MRI used for diabetic foot ulcer?...


c:\Healthcare_Prior_Authorization_AI_Assistant\h1venv\lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[12/12] Does Medicare cover MRI of the spine?...


c:\Healthcare_Prior_Authorization_AI_Assistant\h1venv\lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



Evaluation complete!


In [5]:
# Compute metrics
in_scope = df[df['expected_policy'] != 'DECLINE']
out_of_scope = df[df['expected_policy'] == 'DECLINE']

print("="*60)
print("EVALUATION METRICS")
print("="*60)

# Retrieval metrics (in-scope only)
policy_routing_acc = in_scope['correct_policy_in_top5'].sum() / len(in_scope) * 100
page_retrieval_acc = in_scope['correct_page_in_top5'].sum() / len(in_scope) * 100

print(f"\nIn-scope questions: {len(in_scope)}")
print(f"  Policy Routing Accuracy (correct policy in top-5): {policy_routing_acc:.1f}%")
print(f"  Page Retrieval@5 (correct page in top-5): {page_retrieval_acc:.1f}%")

# Behavior metrics
in_scope_answered = (~in_scope['declined']).sum()
in_scope_answered_pct = in_scope_answered / len(in_scope) * 100
print(f"  Answered (not declined): {in_scope_answered}/{len(in_scope)} ({in_scope_answered_pct:.1f}%)")

# Out-of-scope
if len(out_of_scope) > 0:
    print(f"\nOut-of-scope questions: {len(out_of_scope)}")
    declined_correctly = out_of_scope['declined'].sum()
    print(f"  Correctly Declined: {declined_correctly}/{len(out_of_scope)} ({declined_correctly/len(out_of_scope)*100:.1f}%)")

# Overall
overall_ok = df['expected_behavior_ok'].sum() / len(df) * 100
print(f"\nOverall Expected Behavior: {overall_ok:.1f}%")

print("\n" + "="*60)
print("PER-QUESTION BREAKDOWN")
print("="*60)
for _, row in df.iterrows():
    status = "✅" if row['expected_behavior_ok'] else "❌"
    print(f"{status} [{row['expected_policy']}] {row['question'][:60]}")
    if row['expected_policy'] != 'DECLINE':
        print(f"    Policy in top-5: {row['correct_policy_in_top5']}, Page in top-5: {row['correct_page_in_top5']}")
        print(f"    Declined: {row['declined']}")

EVALUATION METRICS

In-scope questions: 11
  Policy Routing Accuracy (correct policy in top-5): 100.0%
  Page Retrieval@5 (correct page in top-5): 72.7%
  Answered (not declined): 9/11 (81.8%)

Out-of-scope questions: 1
  Correctly Declined: 1/1 (100.0%)

Overall Expected Behavior: 83.3%

PER-QUESTION BREAKDOWN
✅ [0236] What are the medical necessity criteria for MRI of the spine
    Policy in top-5: True, Page in top-5: True
    Declined: False
❌ [0236] When is dynamic-kinetic MRI considered experimental?
    Policy in top-5: True, Page in top-5: False
    Declined: True
❌ [0236] When is MRI not medically necessary for spine trauma?
    Policy in top-5: True, Page in top-5: False
    Declined: True
✅ [0673] When is knee arthroscopy medically necessary?
    Policy in top-5: True, Page in top-5: True
    Declined: False
✅ [0673] What are the criteria for meniscal repair?
    Policy in top-5: True, Page in top-5: True
    Declined: False
✅ [0520] What are the indications for cardiac MRI?

In [6]:
# Debug the 2 failures
failing_queries = [
    "When is dynamic-kinetic MRI considered experimental?",
    "When is MRI not medically necessary for spine trauma?"
]

for q in failing_queries:
    print("="*80)
    print(f"Q: {q}")
    print("-"*80)
    retrieved = vectorstore.similarity_search(q, k=5)
    for i, doc in enumerate(retrieved):
        print(f"  Rank {i+1}: Policy {doc.metadata['policy_number']}, Page {doc.metadata['page']}")

Q: When is dynamic-kinetic MRI considered experimental?
--------------------------------------------------------------------------------
  Rank 1: Policy 0236, Page 18
  Rank 2: Policy 0236, Page 18
  Rank 3: Policy 0171, Page 19
  Rank 4: Policy 0520, Page 63
  Rank 5: Policy 0171, Page 33
Q: When is MRI not medically necessary for spine trauma?
--------------------------------------------------------------------------------
  Rank 1: Policy 0236, Page 2
  Rank 2: Policy 0236, Page 44
  Rank 3: Policy 0236, Page 14
  Rank 4: Policy 0236, Page 11
  Rank 5: Policy 0236, Page 41


In [7]:
# Save results
results_path = project_root / "data" / "evaluation_results.csv"
df.to_csv(results_path, index=False)
print(f"Saved to: {results_path}")

Saved to: c:\Healthcare_Prior_Authorization_AI_Assistant\data\evaluation_results.csv
